# Step 14: ML models + ANN on unseen conversations
Whole conversations are held out (3 folds). This fixes the data leakage of a random split.
*Same as `src/14_session_eval.py`.*

Go to the project folder.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Load the data.** The 5 bookkeeping columns describe the recording, not the person, so they are dropped.

In [ ]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")

# the first 5 face columns are bookkeeping (frame number, face id, time, confidence, success)
bookkeeping = ["face_0", "face_1", "face_2", "face_3", "face_4"]
cols = []
for c in df.columns:
    if (c.startswith("voice_") or c.startswith("face_")) and c not in bookkeeping:
        cols.append(c)

X = df[cols].fillna(0).to_numpy("float32")      # features
y = df["engagement"].to_numpy("float32")        # labels
print(X.shape)

**The 3 folds.** Each fold holds out 3 whole conversations (one per language).

In [ ]:
# the 3 sessions of each language
english = sorted(df[df["language"] == "English"]["session"].unique())
french = sorted(df[df["language"] == "French"]["session"].unique())
german = sorted(df[df["language"] == "German"]["session"].unique())

# fold 1 tests on the 1st session of each language, fold 2 on the 2nd, fold 3 on the 3rd
folds = []
for k in range(3):
    folds.append([english[k], french[k], german[k]])
print(folds)

**The models.** A new copy of each model is made for every fold.

In [ ]:
def make_models():
    return {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "GradientBoosting": HistGradientBoostingRegressor(max_iter=300, random_state=42),
        "XGBoost": XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                                colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1),
    }


def make_ann():
    keras.utils.set_random_seed(42)
    model = keras.Sequential([
        layers.Input(shape=(X.shape[1],)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    return model

**Train and test on every fold.** The scaler learns only from the training sessions. For the ANN the scaled values are also limited to -5..5 (a few extreme values break neural networks).

In [ ]:
scores = {"LinearRegression": [], "RandomForest": [], "GradientBoosting": [], "XGBoost": [], "ANN (MLP)": []}

for test_sessions in folds:
    test = df["session"].isin(test_sessions).to_numpy()
    train = ~test

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train])
    X_test = scaler.transform(X[test])

    for name, model in make_models().items():
        model.fit(X_train, y[train])
        scores[name].append(ccc(y[test], model.predict(X_test)))

    ann = make_ann()
    ann.fit(np.clip(X_train, -5, 5), y[train], epochs=30, batch_size=128, verbose=0)
    scores["ANN (MLP)"].append(ccc(y[test], ann.predict(np.clip(X_test, -5, 5), verbose=0).flatten()))
    print("fold done:", test_sessions)

**Results** (mean over 3 folds) next to the result saved by the script.

In [ ]:
saved = pd.read_csv("reports/session_flat.csv").groupby("model")["ccc"].mean()
for name in scores:
    print(name.ljust(18), "mean", round(np.mean(scores[name]), 3), " std", round(np.std(scores[name], ddof=1), 3),
          " saved", round(saved[name], 3))